In [2]:
import requests
import pandas as pd
import time

BASE_URL = "https://api.deezer.com"

ARTISTS = [
    "Rulo y la contrabanda",
    "El duende callejero",
    "Arde Bogota",
    "Mr Kilombo",
    "Taburete",
    "La Plazuela",
    "Veintiuno",
    "Ojete Calor",
    "Travis Birds"
]

def buscar_artista (name): 
    url = f"{BASE_URL}/search/artist" #define url con una base común para todas incluida en variable BASE_URL y un añadido para esta función
    params = {"q": name, "limit": 1} #diccionario para definir la busqueda, el nombre lo buscaremos desde su clave q(definida por deezer) limite 1 para que solo exporte 1 artista
    r = requests.get(url, params=params) # llamar a la url con los parámetros facilitados
    data = r.json()
    return data["data"][0] 

def buscar_albumes (id): #lo usamos para encontrar los album
    url = f"{BASE_URL}/artist/{id}/albums" #con la url base y el id obtenido en la función buscar_artista
    r = requests.get(url)
    data = r.json()
    return data["data"]

def buscar_nbtracks(id): #lo usamos para buscar temas dentro de los album, funciona con el id de album no de artista
    url = f"{BASE_URL}/album/{id}"
    r = requests.get(url)
    data = r.json()
    return data["nb_tracks"]

In [3]:
results = []

for artista in ARTISTS: #si artista está en la lista que creamos al principio
    info_artista = buscar_artista (artista) #buscamos al artista llamando a la función y creamos variable
    print(f"Procesando: {artista}...") #ameniza un poco la espera enseñándonos a quien está procesando
    artist_id = info_artista ["id"] #extraemos su id con la variable anterior e indicando la clave que buscamos al ser diccionario
    albumes = buscar_albumes (artist_id) #llamamos a la funcion obtener_albumes con el id de artista obtenido en la anterior y creamos variable

    total = 0
    for album in albumes:
        album_id = album["id"] #buscamos el id llamando a la clave
        tracks = buscar_nbtracks (album_id) #buscamos el numero de canciones llamando a buscar_nbtracks con la id de album obtenida en el anterior
        total = total + tracks #sumamos a total el numero de canciones obtenidas
        time.sleep(0.1) #adicional, como deezer tiene un max de llamadas por seg, hacemos que python haga una pausa para no sobrecargar la API

    results.append({ #creamos un diccionario con los resultados obtenidos por artista
        "artista": artista,
        "total_canciones": total
    })
    print(f"✅ {artista}: {total} canciones") #nos indica que artista ha procesado ya y el total de canciones

df = pd.DataFrame(results) #convertir a tabla con pandas, va fuera del bucle porque si lo ponemos dentro imprime toda la lista con cada artista
df["supera_50"] = df["total_canciones"].apply(lambda x: "✅ Sí" if x >= 50 else "❌ No") 
#df supera 50 añade columna con ese nombre, apply asigna funcion a cada parte, lambda es una función rápida para el if/else
print(df)

Procesando: Rulo y la contrabanda...
✅ Rulo y la contrabanda: 146 canciones
Procesando: El duende callejero...
✅ El duende callejero: 44 canciones
Procesando: Arde Bogota...
✅ Arde Bogota: 44 canciones
Procesando: Mr Kilombo...
✅ Mr Kilombo: 66 canciones
Procesando: Taburete...
✅ Taburete: 98 canciones
Procesando: La Plazuela...
✅ La Plazuela: 70 canciones
Procesando: Veintiuno...
✅ Veintiuno: 119 canciones
Procesando: Ojete Calor...
✅ Ojete Calor: 52 canciones
Procesando: Travis Birds...
✅ Travis Birds: 49 canciones
                 artista  total_canciones supera_50
0  Rulo y la contrabanda              146      ✅ Sí
1    El duende callejero               44      ❌ No
2            Arde Bogota               44      ❌ No
3             Mr Kilombo               66      ✅ Sí
4               Taburete               98      ✅ Sí
5            La Plazuela               70      ✅ Sí
6              Veintiuno              119      ✅ Sí
7            Ojete Calor               52      ✅ Sí
8        